# PointDiT on Kaggle (T4)
End-to-end run of PointDiT-L (512x512) on a 10-image subset of the Booster benchmark:
1. inference on the images (depth maps + point clouds), and
2. zero-shot evaluation against Booster ground truth (Rel / delta1 / boundary F1).

**Before running:** Settings -> Accelerator **GPU T4**, Settings -> **Internet on**, and add a Kaggle secret
`HF_TOKEN` (Add-ons -> Secrets) holding a Hugging Face token whose account was granted access to
[facebook/dinov3-vitl16-pretrain-lvd1689m](https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m).
A read-only token is enough.

## 1. Setup environment & clone repositories
Kaggle already ships PyTorch / torchvision with CUDA, so only the remaining requirements are installed.
`huggingface_hub` is deliberately **not** upgraded: the newest major version drops the `huggingface-cli`
command and breaks Kaggle's preinstalled `transformers`. Everything below uses the Python API instead.

In [1]:
import os, subprocess, torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - enable the T4 accelerator')

WORK = '/kaggle/working'
REPO = f'{WORK}/pointdit'
if not os.path.isdir(REPO):
    !git clone https://github.com/Sainava/Sai-pointdit.git {REPO}
if not os.path.isdir(f'{REPO}/third_party/dinov3'):
    !git clone https://github.com/facebookresearch/dinov3.git {REPO}/third_party/dinov3
!pip install -q -r {REPO}/requirements.txt safetensors
%cd {REPO}

torch 2.10.0+cu128 | CUDA True | Tesla T4
Cloning into '/kaggle/working/pointdit'...
remote: Enumerating objects: 160, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 160 (delta 9), reused 10 (delta 7), pack-reused 143 (from 1)
Receiving objects: 100% (160/160), 3.60 MiB | 10.96 MiB/s, done.
Resolving deltas: 100% (83/83), done.
Cloning into '/kaggle/working/pointdit/third_party/dinov3'...
remote: Enumerating objects: 660, done.
remote: Counting objects: 100% (327/327), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 660 (delta 195), reused 155 (delta 155), pack-reused 333 (from 1)
Receiving objects: 100% (660/660), 12.99 MiB | 23.01 MiB/s, done.
Resolving deltas: 100% (269/269), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.0/750.0 kB 10.5 MB/s eta 0:00:0000:010:01
/kaggle/working/pointdit


## 2. Authenticate with Hugging Face (Kaggle secret)
Setting `HF_TOKEN` in the environment is all `huggingface_hub` needs; nothing is written to disk or echoed
in the cell output.

In [2]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')  # change the name if your secret differs

from huggingface_hub import whoami
print('Logged in to Hugging Face as', whoami()['name'])

Logged in to Hugging Face as Sainava


## 3. Download PointDiT-L 512 weights

In [3]:
from huggingface_hub import hf_hub_download
CKPT_NAME = 'pointditl-512-mixdata-nodinov3-240c1a4f.pth'
os.makedirs('pretrained/dinov3', exist_ok=True)
ckpt = hf_hub_download('haofeixu/pointdit', CKPT_NAME, local_dir='pretrained')
print(ckpt, f'{os.path.getsize(ckpt) / 1e9:.2f} GB')

pointditl-512-mixdata-nodinov3-240c1a4f.(…):   0%|          | 0.00/3.75G [00:00<?, ?B/s]

pretrained/pointditl-512-mixdata-nodinov3-240c1a4f.pth 3.75 GB


## 4. DINOv3 ViT-L/16 weights (gated): download from Hugging Face and convert
PointDiT loads the encoder through `torch.hub` from the DINOv3 repo and expects Meta's original checkpoint
`pretrained/dinov3/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth`. The Hugging Face repo only hosts the
`transformers` version (`model.safetensors`, different key names, q/k/v split), so the cell below remaps it
to the original layout and verifies it by strict-loading into the DINOv3 model.

*Alternative:* if you also got Meta's download e-mail, you can skip the conversion and fetch the `.pth`
directly with `!wget -O pretrained/dinov3/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth "<URL from the e-mail>"`.

In [4]:
import re
from safetensors.torch import load_file

DINO_PTH = 'pretrained/dinov3/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'

def hf_dinov3_to_hub(hf_sd, template_sd):
    """Map a transformers DINOv3ViTModel state dict onto the torch.hub (Meta) layout."""
    used = set()
    norm = {re.sub(r'^(dinov3\.|backbone\.)', '', k): k for k in hf_sd}
    def g(name):
        used.add(norm[name]); return hf_sd[norm[name]]
    out = {
        'cls_token': g('embeddings.cls_token'),
        'storage_tokens': g('embeddings.register_tokens'),
        'mask_token': g('embeddings.mask_token').reshape(template_sd['mask_token'].shape),
        'patch_embed.proj.weight': g('embeddings.patch_embeddings.weight'),
        'patch_embed.proj.bias': g('embeddings.patch_embeddings.bias'),
        'norm.weight': g('norm.weight'), 'norm.bias': g('norm.bias'),
        # RoPE periods are a deterministic buffer, not stored by transformers
        'rope_embed.periods': template_sd['rope_embed.periods'].clone(),
    }
    # layer prefix differs across transformers versions: 'layer.N.' or 'model.layer.N.'
    pre = 'model.layer' if any(k.startswith('model.layer.') for k in norm) else 'layer'
    layer_ids = sorted({int(m.group(1)) for k in norm if (m := re.match(rf'^{re.escape(pre)}\.(\d+)\.', k))})
    for i in layer_ids:
        h, b = f'{pre}.{i}.', f'blocks.{i}.'
        for a, c in [('norm1', 'norm1'), ('norm2', 'norm2'), ('attention.o_proj', 'attn.proj'),
                     ('mlp.up_proj', 'mlp.fc1'), ('mlp.down_proj', 'mlp.fc2')]:
            out[b + c + '.weight'] = g(h + a + '.weight')
            out[b + c + '.bias'] = g(h + a + '.bias')
        out[b + 'ls1.gamma'] = g(h + 'layer_scale1.lambda1')
        out[b + 'ls2.gamma'] = g(h + 'layer_scale2.lambda1')
        q_w, k_w, v_w = (g(h + f'attention.{p}_proj.weight') for p in 'qkv')
        q_b, v_b = g(h + 'attention.q_proj.bias'), g(h + 'attention.v_proj.bias')
        out[b + 'attn.qkv.weight'] = torch.cat([q_w, k_w, v_w], 0)
        # DINOv3 has no key bias: the original stores a zero k-bias plus a 1/0/1 mask
        out[b + 'attn.qkv.bias'] = torch.cat([q_b, torch.zeros_like(q_b), v_b], 0)
        out[b + 'attn.qkv.bias_mask'] = torch.cat([torch.ones_like(q_b), torch.zeros_like(q_b), torch.ones_like(v_b)], 0)
    unused = set(hf_sd) - used
    assert not unused, f'unmapped HF keys: {sorted(unused)[:10]}'
    assert set(out) == set(template_sd), f'key mismatch: {sorted(set(out) ^ set(template_sd))[:10]}'
    for k in out:
        assert out[k].shape == template_sd[k].shape, (k, out[k].shape, template_sd[k].shape)
    return {k: v.float().contiguous() for k, v in out.items()}

if not os.path.isfile(DINO_PTH):
    st = hf_hub_download('facebook/dinov3-vitl16-pretrain-lvd1689m', 'model.safetensors',
                         local_dir='/kaggle/tmp/dinov3_hf')
    template = torch.hub.load('third_party/dinov3', 'dinov3_vitl16', source='local', pretrained=False)
    torch.save(hf_dinov3_to_hub(load_file(st), template.state_dict()), DINO_PTH)
    del template
    os.remove(st)  # free ~1.2 GB

# Verify exactly the way PointDiT loads it (strict, weights_only)
enc = torch.hub.load('third_party/dinov3', 'dinov3_vitl16', source='local', pretrained=False)
enc.load_state_dict(torch.load(DINO_PTH, map_location='cpu', weights_only=True), strict=True)
del enc
print('DINOv3 ViT-L/16 weights ready:', DINO_PTH, f'{os.path.getsize(DINO_PTH) / 1e9:.2f} GB')

model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

DINOv3 ViT-L/16 weights ready: pretrained/dinov3/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth 1.21 GB


## 5. Download Booster and pick a 10-image subset
Booster (38 scenes, ~19 MB) ships with ground-truth depth and intrinsics, so the same 10 scenes are used for
both the inference demo and the evaluation.

In [5]:
import shutil
os.makedirs('datasets/eval', exist_ok=True)
hf_hub_download('haofeixu/pointdit-eval', 'Booster.zip', repo_type='dataset', local_dir='datasets/eval')
!cd datasets/eval && unzip -qo Booster.zip

SUBSET = ['Motorcycle', 'Microwave', 'Bedroom', 'Mirror', 'Fridge', 'DogHouse', 'Shower', 'Door', 'TV', 'Bathroom']

# (a) plain images for wild inference
os.makedirs('datasets/wild/Booster_subset', exist_ok=True)
# (b) the same scenes with GT, laid out like a benchmark dataset for evaluation
shutil.rmtree('datasets/eval_subset', ignore_errors=True)
for name in SUBSET:
    src = f'datasets/eval/Booster/{name}'
    assert os.path.isfile(f'{src}/image.jpg'), f'missing {src}/image.jpg'
    shutil.copy(f'{src}/image.jpg', f'datasets/wild/Booster_subset/{name}.jpg')
    shutil.copytree(src, f'datasets/eval_subset/Booster/{name}')
print(len(os.listdir('datasets/wild/Booster_subset')), 'images ready')

Booster.zip:   0%|          | 0.00/19.0M [00:00<?, ?B/s]

10 images ready


## 6. Inference on the 10 images
`main.py` is called directly: `scripts/demo_l_512.sh` hard-codes `IMAGE_DIR=assets/demo`, so an `IMAGE_DIR=...`
prefix would be ignored and it would only process the bundled sample image. Same flags as the script, plus
`--num_workers 2` (Kaggle has 4 CPU cores; the default is 16).

Results: `generation/pretrained-pointditl-512-mixdata-nodinov3-240c1a4f/Booster_subset/` (`*_depth.png`, `*_pointcloud.ply`).

In [6]:
!DINOV3_WEIGHTS_DIR=pretrained/dinov3 python main.py \
  --model PointDiT-L/16 --feature_embedding_type dinov3_vitl16 --proj_dropout 0.0 \
  --evaluate_gen --eval_wild_images --eval_wild_images_dir datasets/wild/Booster_subset \
  --img_size 512 --num_sampling_steps 3 --num_workers 2 \
  --pretrained pretrained/{CKPT_NAME}

[Warning] full-frame wild inference requires batch_size=1, overriding batch_size=128 to 1
[WildImagesDataset] Found 10 images in datasets/wild/Booster_subset
Test dataset: WildImagesDataset(Booster_subset, 10 images)
Downloading: "file:///kaggle/working/pointdit/pretrained/dinov3/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth" to /root/.cache/torch/hub/checkpoints/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
100%|██████████████████████████████████████| 1.13G/1.13G [00:00<00:00, 2.51GB/s]
Loaded pretrained DINOv3 vitl16 from pretrained/dinov3/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
Number of trainable parameters: 463.06M
Number of non-trainable parameters: 308.40M
Loaded EMA weights in place for evaluation
Load pretrained model from pretrained/pointditl-512-mixdata-nodinov3-240c1a4f.pth
/usr/lib/python3.12/contextlib.py:137: UserWarning: CUDA reports that you have 2 available devices, and you have used fork_rng without explicitly specifying which devices are being used. For safety, w

## 7. Evaluation against ground truth on the same 10 scenes
Same settings as `scripts/eval_l_512.sh`, restricted to the Booster subset. Note the paper numbers are over
all 7 benchmarks (3,444 samples), so 10 Booster scenes will not reproduce them exactly.

In [7]:
!DINOV3_WEIGHTS_DIR=pretrained/dinov3 python main.py \
  --model PointDiT-L/16 --feature_embedding_type dinov3_vitl16 --proj_dropout 0.0 \
  --evaluate_gen \
  --img_size 512 --eval_depth_resize_height 512 --num_sampling_steps 3 --num_workers 2 \
  --eval_depth_dataset --eval_depth_data_root datasets/eval_subset \
  --eval_depth_dataset_name Booster --eval_boundary_datasets Booster \
  --pretrained pretrained/{CKPT_NAME}

[EvalDepthDataset] Will resize shortest side to 512
[EvalDepthDataset] Using center crop of size (512, 512)
[EvalDepthDataset] Discovered 10 samples from Booster via glob
Test dataset: <dataloader.eval_depth.EvalDepthDataset object at 0x7ed1a047be00>
Loaded pretrained DINOv3 vitl16 from pretrained/dinov3/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
Number of trainable parameters: 463.06M
Number of non-trainable parameters: 308.40M
Loaded EMA weights in place for evaluation
Load pretrained model from pretrained/pointditl-512-mixdata-nodinov3-240c1a4f.pth
/usr/lib/python3.12/contextlib.py:137: UserWarning: CUDA reports that you have 2 available devices, and you have used fork_rng without explicitly specifying which devices are being used. For safety, we initialize *every* CUDA device by default, which can be quite slow if you have a lot of CUDAs. If you know that you are only making use of a few CUDA devices, set the environment variable CUDA_VISIBLE_DEVICES or the 'devices' keyword argu

## 8. Zip results

In [8]:
!cd {REPO} && zip -qr {WORK}/pointdit_results.zip generation/
print('Download /kaggle/working/pointdit_results.zip from the Output panel.')

Download /kaggle/working/pointdit_results.zip from the Output panel.
